In [ ]:
from secfsn_plus_models.notebook_entry import init_notebook
init_notebook()


# Layer 04 Demo: `secfsn_plus_models`

This notebook is a short video-ready walkthrough of **Layer 04** in the SECFSN stack.

Canonical stack order:

1. `secfsn`
2. `secfsn_plus`
3. `secfsn_plus_structural`
4. `secfsn_plus_models`
5. `secfsn_plus_decisionsystems`

Layer 04 consumes a prepared feature matrix, creates labels, trains a model, evaluates the resulting predictions, and packages model outputs for downstream consumption.

This notebook uses **synthetic data only**. It does not load raw SEC data and does not make final trading decisions.

## Imports

We reuse the repository's tested APIs rather than rebuilding the pipeline by hand. That keeps the demo aligned with the package's actual Layer 04 behavior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from secfsn_plus_models.automation import generate_synthetic_feature_matrix
from secfsn_plus_models.dataset import build_model_dataset
from secfsn_plus_models.evaluation import evaluate_cross_sectional_model
from secfsn_plus_models.labels import compute_next_period_return
from secfsn_plus_models.models.linear import LinearRegressionModel
from secfsn_plus_models.notebook_utils import (
    preview_aligned_dataset,
    summarize_dataset,
    summarize_period_coverage,
    summarize_period_metrics,
    summarize_split_artifacts,
)
from secfsn_plus_models.outputs import build_model_output_frame
from secfsn_plus_models.splits import time_based_split
from secfsn_plus_models.trainer import train_and_predict

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


## Build A Synthetic Feature Matrix

In production, Layer 04 consumes prepared features from Layers 02 and 03. For a fast and stable demo, we generate a deterministic synthetic panel with a canonical `MultiIndex(cik, period)`.

In [ ]:
features_df = generate_synthetic_feature_matrix(price_column="price", seed=20240323)
features_df.head(8)


## Inspect The Layer 04 Input Shape

The key point is that Layer 04 expects a prepared company-period panel, not raw SEC filing data.

In [ ]:
dataset_summary = summarize_dataset(features_df)
period_coverage = summarize_period_coverage(features_df)

dataset_summary, period_coverage


## Create Labels / Targets

This repository includes a helper that computes next-period forward returns from a price-like column. That gives us a simple modeling target without introducing any raw data loading.

In [ ]:
labels = compute_next_period_return(features_df, "price")
labels.head(8)


## Preview The Aligned Modeling Dataset

Next we align the features and labels, then preserve selected metadata columns separately.

In [ ]:
preview_aligned_dataset(features_df, labels, limit=8)


## Build The Model Dataset And Split It In Time

Layer 04 is responsible for turning the prepared panel into a trainable feature matrix and an aligned target series. We exclude `price` and `sector_code` from the feature matrix here so they remain metadata rather than direct model inputs.

In [ ]:
X, y, meta = build_model_dataset(features_df, labels, exclude_columns=["price", "sector_code"])
X_train, y_train, X_val, y_val, X_test, y_test = time_based_split(X, y, train_end_period="2020Q4")

split_summary = summarize_split_artifacts(
    type(
        "ArtifactsProxy",
        (),
        {
            "X": X,
            "meta": meta,
            "X_train": X_train,
            "X_test": X_test,
        },
    )()
)

split_summary


## Train A Simple Deterministic Model

For a video walkthrough, a linear model is ideal because it is fast, deterministic, and easy to explain.

In [ ]:
model = LinearRegressionModel()
predictions = train_and_predict(model, X_train, y_train, X_test)
predictions.head(8)


## Evaluate The Predictions

These utilities summarize cross-sectional prediction quality. The goal here is not to claim production alpha; it is to show the standard Layer 04 evaluation path.

In [ ]:
evaluation_summary = evaluate_cross_sectional_model(y_test, predictions)
period_metrics = summarize_period_metrics(y_test, predictions)

evaluation_summary, period_metrics


## Package Layer 04 Outputs For Layer 05

This is the key handoff step. We package predictions into a canonical downstream shape that preserves the `(cik, period)` index and adds:

- `model_prediction`
- `model_expected_return`
- `model_risk_probability`

`model_risk_probability` here is a **downstream handoff signal** derived from ranked model output. It is **not** a calibrated fraud probability or a final decision by itself.

In [ ]:
model_outputs = build_model_output_frame(
    predictions,
    higher_prediction_more_risky=False,
)
model_outputs.head(10)


## Optional Quick Plot

A lightweight scatter plot can help during a recording, but it is intentionally simple so the notebook stays fast and stable.

In [ ]:
plot_frame = pd.DataFrame({
    "actual": y_test,
    "prediction": predictions,
}).dropna()

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(plot_frame["prediction"], plot_frame["actual"], alpha=0.7)
ax.set_title("Layer 04: Prediction vs Actual")
ax.set_xlabel("model_prediction")
ax.set_ylabel("next_period_return")
ax.axhline(0.0, color="gray", linewidth=1)
ax.axvline(0.0, color="gray", linewidth=1)
plt.show()


## Closing Notes

Layer 04 does:

- consume prepared feature matrices
- create labels
- train and evaluate models
- package model outputs for downstream use

Layer 04 does **not**:

- parse raw SEC filings
- derive canonical accounting ratios
- generate final long/short/risk decisions

Those final deterministic decision rules belong in **Layer 05: `secfsn_plus_decisionsystems`**.